In [4]:
!playwright install

|                                                                                |   0% of 148.9 MiB
|■■■■■■■■                                                                        |  10% of 148.9 MiB
|■■■■■■■■■■■■■■■■                                                                |  20% of 148.9 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■                                                        |  30% of 148.9 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■                                                |  40% of 148.9 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■                                        |  50% of 148.9 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■                                |  60% of 148.9 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■                        |  70% of 148.9 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■                |  80% of 148.9 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■        |  90% of 

In [25]:
import os
import io
import json
import time
import base64
import asyncio
from typing import List, Dict, Any, Optional
import requests
from bs4 import BeautifulSoup
from PIL import Image
import numpy as np
import re

# 2f17994f94c793e49ad738e91889b37c5d34c3e0ad0c3a3ea7d0a49e9de978e2
SERPAPI_KEY = os.getenv(
    "SERPAPI_KEY", "2f17994f94c793e49ad738e91889b37c5d34c3e0ad0c3a3ea7d0a49e9de978e2"
)
PLAYWRIGHT_TIMEOUT = 10000  # ms

In [ ]:
# Fetching Page information
def fetch_page(url: str, timeout=15) -> Dict[str, Any]:
    """Multi-strategy page fetcher that works without Playwright."""
    results = {"url": url, "fetched_with": None, "html": "", "error": None}

    # Strategy 1: Standard requests with browser-like headers
    browser_headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml",
        "Accept-Language": "en-US,en;q=0.9",
        "Connection": "keep-alive",
        "Upgrade-Insecure-Requests": "1",
        "Sec-Fetch-Dest": "document",
        "Sec-Fetch-Mode": "navigate",
        "Sec-Fetch-Site": "none",
    }

    try:
        session = requests.Session()
        # First make a HEAD request to check content type
        head = session.head(
            url, timeout=timeout / 2, headers=browser_headers, allow_redirects=True
        )
        content_type = head.headers.get("Content-Type", "")

        # Proceed with GET if it looks like HTML content
        if "text/html" in content_type or not content_type:
            response = session.get(url, timeout=timeout, headers=browser_headers)
            response.raise_for_status()
            results["html"] = response.text
            results["fetched_with"] = "requests"
            return results
    except Exception as e:
        results["error"] = f"Standard request failed: {str(e)}"

    # Strategy 2: Try with cloudscraper (handles Cloudflare protection)
    try:
        import cloudscraper

        scraper = cloudscraper.create_scraper(
            browser={"browser": "chrome", "platform": "windows"}
        )
        response = scraper.get(url, timeout=timeout)
        results["html"] = response.text
        results["fetched_with"] = "cloudscraper"
        return results
    except ImportError:
        results["error"] = f"{results.get('error', '')} | cloudscraper not installed"
    except Exception as e:
        results["error"] = f"{results.get('error', '')} | cloudscraper failed: {str(e)}"

    # Strategy 3: Try requests-html for basic JS rendering
    try:
        from requests_html import HTMLSession

        session = HTMLSession()
        response = session.get(url, timeout=timeout, headers=browser_headers)
        response.raise_for_status()

        # Simple render of JavaScript
        try:
            response.html.render(timeout=timeout, sleep=1)
        except:
            pass  # Continue with what we have if render fails

        results["html"] = response.html.html
        results["fetched_with"] = "requests_html"
        return results
    except ImportError:
        results["error"] = f"{results.get('error', '')} | requests-html not installed"
    except Exception as e:
        results["error"] = (
            f"{results.get('error', '')} | requests-html failed: {str(e)}"
        )

    return results

In [33]:
def extract_text_and_images(html: str, base_url: str) -> Dict[str, Any]:
    """
    Enhanced extractor for detailed product information:
    - Basic: title, meta tags, JSON-LD, images
    - Content: paragraphs, bullet points, description blocks
    - Specifications: detailed product attributes, tech specs, size/weight
    - Materials & Care: fabric composition, washing instructions
    - Variant data: colors, sizes, patterns available
    """
    soup = BeautifulSoup(html, "html.parser")

    # Title and Metas
    title = soup.title.string.strip() if soup.title and soup.title.string else ""
    metas = {}
    for m in soup.find_all("meta"):
        if m.get("name") and m.get("content"):
            metas[m["name"].lower()] = m["content"]
        if m.get("property") and m.get("content"):
            metas[m["property"].lower()] = m["content"]
    description_meta = metas.get("og:description") or metas.get("description")

    # JSON-LD extraction (enhanced to capture nested structures)
    jsonld = []
    for tag in soup.find_all("script", type="application/ld+json"):
        try:
            raw = tag.string or tag.get_text() or ""
            parsed = json.loads(raw)
            if isinstance(parsed, list):
                jsonld.extend(parsed)
            else:
                jsonld.append(parsed)
        except Exception:
            try:
                parts = raw.strip()
                parsed = json.loads(f"[{parts}]")
                jsonld.extend(parsed)
            except Exception:
                continue

    # Try to extract product data from JavaScript variables
    js_data = {}
    for script in soup.find_all("script"):
        script_text = script.string or ""
        if not script_text:
            continue

        if (
            "window.__PRELOADED_STATE__" in script_text
            or "window.__INITIAL_STATE__" in script_text
        ):
            try:
                matches = re.search(
                    r"window\.__(?:PRELOADED|INITIAL)_STATE__\s*=\s*({.+?});",
                    script_text,
                    re.DOTALL,
                )
                if matches:
                    js_data["state"] = json.loads(matches.group(1))
            except:
                pass

        if (
            "var product" in script_text.lower()
            or "window.product" in script_text.lower()
        ):
            try:
                matches = re.search(
                    r"(?:var|window)\.product(?:Data|Json|Info)?\s*=\s*({.+?});",
                    script_text,
                    re.DOTALL,
                )
                if matches:
                    js_data["product"] = json.loads(matches.group(1))
            except:
                pass

    # Paragraphs - extended to get more text content
    paras = [p.get_text().strip() for p in soup.find_all("p") if p.get_text().strip()]

    # Description blocks (more comprehensive)
    description_blocks = []
    for selector in [
        "#productDescription",
        ".product-description",
        "#product-description",
        ".prod-detail",
        ".prod-info",
    ]:
        elements = soup.select(selector)
        for el in elements:
            text = el.get_text(" ", strip=True)
            if text and len(text) > 20:  # Meaningful content only
                description_blocks.append(text)

    # Bullets - expanded selectors
    bullets = []
    selectors = [
        "#feature-bullets li",
        "#detailBullets_feature_div li",
        "#productDescription li",
        ".a-unordered-list.a-vertical li",
        "#feature-bullets ul li",
        ".product-features li",
        ".product-highlights li",
        ".features-list li",
        ".key-features li",
    ]
    for sel in selectors:
        for li in soup.select(sel):
            txt = li.get_text(" ", strip=True)
            if txt:
                bullets.append(txt)
    bullets = list(dict.fromkeys(bullets))  # dedupe preserving order

    # Product details - extensively enhanced
    product_details = {}

    # helper: merge repeated keys into lists, preserving order and avoiding duplicates
    def add_detail(coll: Dict[str, Any], key: str, val: str):
        if not key:
            return
        # normalize values
        if isinstance(val, str):
            v = val.strip()
        else:
            v = val
        if key in coll:
            cur = coll[key]
            if isinstance(cur, list):
                if v and v not in cur:
                    cur.append(v)
            else:
                if v and v != cur:
                    coll[key] = [cur, v]
        else:
            coll[key] = v

    # Table-based details
    for table in soup.select("table"):
        if table.find("tr"):
            # Get table caption/heading if available
            table_heading = ""
            if table.find("caption"):
                table_heading = table.find("caption").get_text(" ", strip=True)
            elif table.find("th", {"colspan": True}):
                table_heading = table.find("th", {"colspan": True}).get_text(
                    " ", strip=True
                )

            for tr in table.find_all("tr"):
                th = tr.find("th")
                tds = tr.find_all("td")
                if th and tds:
                    key = th.get_text(" ", strip=True)
                    val = " ".join(td.get_text(" ", strip=True) for td in tds)
                    if key:
                        add_detail(product_details, key, val)
                        # Add with section prefix if there was a table heading
                        if table_heading and not key.startswith(table_heading):
                            add_detail(product_details, f"{table_heading} - {key}", val)
                else:
                    cols = tr.find_all("td")
                    if len(cols) >= 2:
                        key = cols[0].get_text(" ", strip=True)
                        val = " ".join(c.get_text(" ", strip=True) for c in cols[1:])
                        if key:
                            add_detail(product_details, key, val)

    # Definition lists
    for dl in soup.find_all(["dl"]):
        dts = dl.find_all("dt")
        dds = dl.find_all("dd")
        for k, v in zip(dts, dds):
            kk = k.get_text(" ", strip=True)
            vv = v.get_text(" ", strip=True)
            if kk:
                add_detail(product_details, kk, vv)

    # Amazon-specific detail bullets
    for li in soup.select(
        "#detailBullets_feature_div li, #productDetails_detailBullets_sections1 li"
    ):
        txt = li.get_text(" ", strip=True)
        if ":" in txt:
            k, v = txt.split(":", 1)
            add_detail(product_details, k.strip(), v.strip())
        elif "-" in txt:
            k, v = txt.split("-", 1)
            add_detail(product_details, k.strip(), v.strip())

    # Key-value pairs from specification tables/divs
    spec_selectors = [
        ".product-specifications",
        ".prod-specs",
        "#specifications",
        "#tech-specs",
        ".specifications",
        ".tech-details",
    ]

    for sel in spec_selectors:
        specs = soup.select(f"{sel} tr, {sel} .spec-row, {sel} .spec-pair")
        for spec in specs:
            # Handle both table rows and div-based specs
            if spec.name == "tr":
                cols = spec.find_all(["th", "td"])
                if len(cols) >= 2:
                    key = cols[0].get_text(" ", strip=True)
                    val = cols[1].get_text(" ", strip=True)
                    if key:
                        add_detail(product_details, key, val)
            else:
                # Div-based layout
                label = spec.select_one(".spec-label, .spec-name, .label")
                value = spec.select_one(".spec-value, .value")
                if label and value:
                    key = label.get_text(" ", strip=True)
                    val = value.get_text(" ", strip=True)
                    if key:
                        add_detail(product_details, key, val)

    # Extract common product attributes more thoroughly
    attributes = {}

    # Colors available
    color_variants = []
    color_elements = soup.select(
        ".color-selector option, .color-swatch, .color-choice, [id*='color-name'], [class*='color-selection']"
    )
    for el in color_elements:
        color = (
            el.get_text(" ", strip=True)
            or el.get("title")
            or el.get("data-color")
            or el.get("value")
        )
        if color and len(color) < 50:  # Sanity check - colors aren't paragraphs
            color_variants.append(color)
    if color_variants:
        attributes["colors"] = list(dict.fromkeys(color_variants))

    # Sizes available
    size_variants = []
    size_elements = soup.select(
        ".size-selector option, .size-swatch, .size-choice, [id*='size-name'], [class*='size-selection']"
    )
    for el in size_elements:
        size = (
            el.get_text(" ", strip=True)
            or el.get("title")
            or el.get("data-size")
            or el.get("value")
        )
        if size and len(size) < 30:  # Sanity check
            size_variants.append(size)
    if size_variants:
        attributes["sizes"] = list(dict.fromkeys(size_variants))

    # Material and fabric - more specific extraction
    material_details = {}
    for key_pattern, selector in [
        (
            "material",
            "[id*='material'], [class*='material'], [id*='fabric'], [class*='fabric']",
        ),
        ("composition", "[id*='composition'], [class*='composition']"),
        (
            "care",
            "[id*='care-instructions'], [class*='care'], [id*='washing'], [class*='washing']",
        ),
    ]:
        elements = soup.select(selector)
        for el in elements:
            text = el.get_text(" ", strip=True)
            if text and len(text) > 3 and len(text) < 500:
                if key_pattern not in material_details:
                    material_details[key_pattern] = []
                material_details[key_pattern].append(text)

    if material_details:
        attributes["material_details"] = material_details

    # Schema Product
    schema_product = {}
    for obj in jsonld:
        if isinstance(obj, dict):
            typ = obj.get("@type") or obj.get("type")
            if typ and ("product" in str(typ).lower()):
                schema_product.update(obj)

    # Extract offers/price information
    price_info = {}

    # From Schema.org
    if schema_product and "offers" in schema_product:
        offers = schema_product["offers"]
        if isinstance(offers, dict):
            price_info["price"] = offers.get("price")
            price_info["currency"] = offers.get("priceCurrency")
            price_info["availability"] = offers.get("availability")
        elif isinstance(offers, list) and offers:
            price_info["price"] = offers[0].get("price")
            price_info["currency"] = offers[0].get("priceCurrency")
            price_info["availability"] = offers[0].get("availability")

    # From page elements
    price_elements = soup.select(
        "[id*='price'], [class*='price']:not(del), .offer-price, .product-price, .current-price"
    )
    for el in price_elements:
        text = el.get_text(" ", strip=True)
        if text and re.search(r"\d", text) and len(text) < 50:
            # Skip "from" prices or ranges for now
            if "from" not in text.lower() and "-" not in text and "to" not in text:
                price_info["displayed_price"] = text
                break

    # Images - enhanced extraction
    images = []
    if metas.get("og:image"):
        images.append(requests.compat.urljoin(base_url, metas.get("og:image")))
    if metas.get("og:image:secure_url"):
        images.append(
            requests.compat.urljoin(base_url, metas.get("og:image:secure_url"))
        )
    schema_img = schema_product.get("image")
    if schema_img:
        if isinstance(schema_img, list):
            for it in schema_img:
                images.append(requests.compat.urljoin(base_url, str(it)))
        else:
            images.append(requests.compat.urljoin(base_url, str(schema_img)))
    link_img = soup.find("link", {"rel": "image_src"})
    if link_img and link_img.get("href"):
        images.append(requests.compat.urljoin(base_url, link_img["href"]))

    # Standard image extraction
    for img in soup.find_all("img"):
        for attr in (
            "src",
            "data-src",
            "data-old-hires",
            "data-srcset",
            "data-lazy-src",
            "data-zoom-image",
            "data-large",
        ):
            val = img.get(attr)
            if val:
                if "," in val and " " in val:
                    parts = [p.strip().split()[0] for p in val.split(",") if p.strip()]
                    for p in parts:
                        images.append(requests.compat.urljoin(base_url, p))
                else:
                    images.append(requests.compat.urljoin(base_url, val))
        srcset = img.get("srcset") or img.get("data-srcset")
        if srcset:
            parts = [p.strip().split()[0] for p in srcset.split(",") if p.strip()]
            for p in parts:
                images.append(requests.compat.urljoin(base_url, p))

    # Look for hidden large images
    for a in soup.find_all("a", href=True):
        href = a.get("href", "")
        if re.search(r"\.(jpe?g|png|webp|gif)(\?|$)", href, re.I):
            images.append(requests.compat.urljoin(base_url, href))

    # Filter and deduplicate images
    images = [i for i in (images or []) if i and not i.endswith((".ico", ".svg"))]
    images = list(dict.fromkeys(images))[:30]  # Keep more images

    # Build text blob with all extracted information
    details_text = " ".join(f"{k}: {v}" for k, v in (product_details or {}).items())
    jsonld_text = " ".join(json.dumps(o) for o in (jsonld or []) if isinstance(o, dict))
    text_blob = "\n".join(
        filter(
            None,
            [
                title,
                description_meta or schema_product.get("description", ""),
                details_text,
                " ".join(bullets or []),
                " ".join(paras or []),
                " ".join(description_blocks or []),
            ],
        )
    )

    # Ensure consistent return types (avoid None) and include enhanced data
    return {
        "title": title or "",
        "metas": metas or {},
        "jsonld": jsonld or [],
        "schema_product": schema_product or {},
        "js_data": js_data or {},
        "paragraphs": paras or [],
        "bullets": bullets or [],
        "description_blocks": description_blocks or [],
        "product_details": product_details or {},
        "attributes": attributes or {},
        "material_details": material_details or {},
        "price_info": price_info or {},
        "images": images or [],
        "text": text_blob or "",
    }

In [35]:
def extract_text_and_images(html: str, base_url: str) -> Dict[str, Any]:
    """
    Enhanced extractor for detailed product information:
    - Basic: title, meta tags, JSON-LD, images
    - Content: paragraphs, bullet points, description blocks
    - Specifications: detailed product attributes, tech specs, size/weight
    - Materials & Care: fabric composition, washing instructions
    - Variant data: colors, sizes, patterns available
    """
    soup = BeautifulSoup(html, "html.parser")

    # Title and Metas
    title = soup.title.string.strip() if soup.title and soup.title.string else ""
    metas = {}
    for m in soup.find_all("meta"):
        if m.get("name") and m.get("content"):
            metas[m["name"].lower()] = m["content"]
        if m.get("property") and m.get("content"):
            metas[m["property"].lower()] = m["content"]
    description_meta = metas.get("og:description") or metas.get("description")

    # JSON-LD extraction (enhanced to capture nested structures)
    jsonld = []
    for tag in soup.find_all("script", type="application/ld+json"):
        try:
            raw = tag.string or tag.get_text() or ""
            parsed = json.loads(raw)
            if isinstance(parsed, list):
                jsonld.extend(parsed)
            else:
                jsonld.append(parsed)
        except Exception:
            try:
                parts = raw.strip()
                parsed = json.loads(f"[{parts}]")
                jsonld.extend(parsed)
            except Exception:
                continue

    # Additional schema data sources specific to Amazon and other retailers
    product_data_scripts = soup.select("#productSchema, #productDetails script")
    for script in product_data_scripts:
        if script.string:
            try:
                # Try to extract JSON from script content
                content = script.string.strip()
                if content.startswith("{") and content.endswith("}"):
                    data = json.loads(content)
                    jsonld.append(data)
            except:
                pass

    # Try to extract product data from JavaScript variables
    js_data = {}
    for script in soup.find_all("script"):
        script_text = script.string or ""
        if not script_text:
            continue

        if (
            "window.__PRELOADED_STATE__" in script_text
            or "window.__INITIAL_STATE__" in script_text
        ):
            try:
                matches = re.search(
                    r"window\.__(?:PRELOADED|INITIAL)_STATE__\s*=\s*({.+?});",
                    script_text,
                    re.DOTALL,
                )
                if matches:
                    js_data["state"] = json.loads(matches.group(1))
            except:
                pass

        # Amazon often stores product data in these variables
        if (
            "var product" in script_text.lower()
            or "window.product" in script_text.lower()
            or "var dataLayer" in script_text
            or "digitalData" in script_text
        ):
            try:
                # Look for product data
                for pattern in [
                    r"(?:var|window)\.product(?:Data|Json|Info)?\s*=\s*({.+?});",
                    r"dataLayer\s*=\s*(\[.+?\]);",
                    r"digitalData\s*=\s*({.+?});",
                ]:
                    matches = re.search(pattern, script_text, re.DOTALL)
                    if matches:
                        data = json.loads(matches.group(1))
                        key = (
                            "product"
                            if "product" in pattern
                            else pattern.split("\\")[0]
                        )
                        js_data[key] = data
            except:
                pass

    # Paragraphs - extended to get more text content
    paras = [p.get_text().strip() for p in soup.find_all("p") if p.get_text().strip()]

    # Description blocks (more comprehensive)
    description_blocks = []
    for selector in [
        "#productDescription",
        ".product-description",
        "#product-description",
        ".prod-detail",
        ".prod-info",
        "#feature-bullets",
        "#productDetails_feature_div",
        "#detail-bullets",
    ]:
        elements = soup.select(selector)
        for el in elements:
            text = el.get_text(" ", strip=True)
            if text and len(text) > 20:  # Meaningful content only
                description_blocks.append(text)

    # Bullets - expanded selectors
    bullets = []
    selectors = [
        "#feature-bullets li",
        "#detailBullets_feature_div li",
        "#productDescription li",
        ".a-unordered-list.a-vertical li",
        "#feature-bullets ul li",
        ".product-features li",
        ".product-highlights li",
        ".features-list li",
        ".key-features li",
        ".a-declarative li",  # Additional Amazon selector
        "#bullets li",  # Additional Amazon selector
        ".detail-bullet-list li",  # Additional Amazon selector
    ]
    for sel in selectors:
        for li in soup.select(sel):
            txt = li.get_text(" ", strip=True)
            if txt:
                bullets.append(txt)
    bullets = list(dict.fromkeys(bullets))  # dedupe preserving order

    # Product details - extensively enhanced
    product_details = {}

    # helper: merge repeated keys into lists, preserving order and avoiding duplicates
    def add_detail(coll: Dict[str, Any], key: str, val: str):
        if not key:
            return
        # normalize values
        if isinstance(val, str):
            v = val.strip()
        else:
            v = val
        if key in coll:
            cur = coll[key]
            if isinstance(cur, list):
                if v and v not in cur:
                    cur.append(v)
            else:
                if v and v != cur:
                    coll[key] = [cur, v]
        else:
            coll[key] = v

    # Look for Amazon's product details tables specifically
    amazon_detail_tables = soup.select(
        "#productDetails_techSpec_section_1, #productDetails_detailBullets_sections1, "
        "#detailBulletsWrapper_feature_div, #technicalSpecifications_section_1, "
        "#tech-specs, #techSpecTable"
    )

    for table_container in amazon_detail_tables:
        # Process all tables within these containers
        for table in table_container.select("table"):
            if table.find("tr"):
                # Get table caption/heading if available
                table_heading = ""
                if table.find("caption"):
                    table_heading = table.find("caption").get_text(" ", strip=True)
                elif table.find("th", {"colspan": True}):
                    table_heading = table.find("th", {"colspan": True}).get_text(
                        " ", strip=True
                    )

                for tr in table.find_all("tr"):
                    th = tr.find("th")
                    tds = tr.find_all("td")
                    if th and tds:
                        key = th.get_text(" ", strip=True)
                        val = " ".join(td.get_text(" ", strip=True) for td in tds)
                        if key:
                            add_detail(product_details, key, val)
                            # Add with section prefix if there was a table heading
                            if table_heading and not key.startswith(table_heading):
                                add_detail(
                                    product_details, f"{table_heading} - {key}", val
                                )
                    else:
                        cols = tr.find_all("td")
                        if len(cols) >= 2:
                            key = cols[0].get_text(" ", strip=True)
                            val = " ".join(
                                c.get_text(" ", strip=True) for c in cols[1:]
                            )
                            if key:
                                add_detail(product_details, key, val)

    # Process any detail rows or detail sections directly (not in tables)
    detail_rows = soup.select(
        ".product-detail-row, .product-attribute, .detail-section"
    )
    for row in detail_rows:
        label = row.select_one(".label, .product-detail-label, .detail-label")
        value = row.select_one(".value, .product-detail-value, .detail-value")

        if label and value:
            key = label.get_text(" ", strip=True)
            val = value.get_text(" ", strip=True)
            if key:
                add_detail(product_details, key, val)

    # Table-based details (general tables beyond Amazon-specific ones)
    for table in soup.select("table"):
        if table.find("tr"):
            # Get table caption/heading if available
            table_heading = ""
            if table.find("caption"):
                table_heading = table.find("caption").get_text(" ", strip=True)
            elif table.find("th", {"colspan": True}):
                table_heading = table.find("th", {"colspan": True}).get_text(
                    " ", strip=True
                )

            for tr in table.find_all("tr"):
                th = tr.find("th")
                tds = tr.find_all("td")
                if th and tds:
                    key = th.get_text(" ", strip=True)
                    val = " ".join(td.get_text(" ", strip=True) for td in tds)
                    if key:
                        add_detail(product_details, key, val)
                        # Add with section prefix if there was a table heading
                        if table_heading and not key.startswith(table_heading):
                            add_detail(product_details, f"{table_heading} - {key}", val)
                else:
                    cols = tr.find_all("td")
                    if len(cols) >= 2:
                        key = cols[0].get_text(" ", strip=True)
                        val = " ".join(c.get_text(" ", strip=True) for c in cols[1:])
                        if key:
                            add_detail(product_details, key, val)

    # Definition lists
    for dl in soup.find_all(["dl"]):
        dts = dl.find_all("dt")
        dds = dl.find_all("dd")
        for k, v in zip(dts, dds):
            kk = k.get_text(" ", strip=True)
            vv = v.get_text(" ", strip=True)
            if kk:
                add_detail(product_details, kk, vv)

    # Amazon-specific detail bullets
    for li in soup.select(
        "#detailBullets_feature_div li, #productDetails_detailBullets_sections1 li, "
        ".detail-bullet-list span.a-list-item, #detailBullets .a-list-item"
    ):
        txt = li.get_text(" ", strip=True)
        if ":" in txt:
            k, v = txt.split(":", 1)
            add_detail(product_details, k.strip(), v.strip())
        elif "-" in txt:
            k, v = txt.split("-", 1)
            add_detail(product_details, k.strip(), v.strip())
        elif "•" in txt:  # Sometimes bullets use this character
            k, v = txt.split("•", 1)
            add_detail(product_details, k.strip(), v.strip())

    # Key-value pairs from specification tables/divs
    spec_selectors = [
        ".product-specifications",
        ".prod-specs",
        "#specifications",
        "#tech-specs",
        ".specifications",
        ".tech-details",
        "#detailBulletsWrapper_feature_div",  # Added Amazon selector
        "#productDetails_techSpec_section_1",  # Added Amazon selector
        "#prodDetails",  # Added Amazon selector
    ]

    for sel in spec_selectors:
        specs = soup.select(
            f"{sel} tr, {sel} .spec-row, {sel} .spec-pair, {sel} .a-spacing-micro"
        )
        for spec in specs:
            # Handle both table rows and div-based specs
            if spec.name == "tr":
                cols = spec.find_all(["th", "td"])
                if len(cols) >= 2:
                    key = cols[0].get_text(" ", strip=True)
                    val = cols[1].get_text(" ", strip=True)
                    if key:
                        add_detail(product_details, key, val)
            else:
                # Div-based layout
                label = spec.select_one(".spec-label, .spec-name, .label, .a-text-bold")
                value = spec.select_one(".spec-value, .value, .a-text-normal")
                if label and value:
                    key = label.get_text(" ", strip=True).rstrip(":")
                    val = value.get_text(" ", strip=True)
                    if key:
                        add_detail(product_details, key, val)

    # Extract common product attributes more thoroughly
    attributes = {}

    # Colors available
    color_variants = []
    color_elements = soup.select(
        ".color-selector option, .color-swatch, .color-choice, [id*='color-name'], "
        "[class*='color-selection'], #variation_color_name li, .colorsprite"
    )
    for el in color_elements:
        color = (
            el.get_text(" ", strip=True)
            or el.get("title")
            or el.get("data-color")
            or el.get("alt")
            or el.get("value")
        )
        if color and len(color) < 50:  # Sanity check - colors aren't paragraphs
            color_variants.append(color)
    if color_variants:
        attributes["colors"] = list(dict.fromkeys(color_variants))

    # Sizes available
    size_variants = []
    size_elements = soup.select(
        ".size-selector option, .size-swatch, .size-choice, [id*='size-name'], "
        "[class*='size-selection'], #variation_size_name li"
    )
    for el in size_elements:
        size = (
            el.get_text(" ", strip=True)
            or el.get("title")
            or el.get("data-size")
            or el.get("value")
        )
        if size and len(size) < 30:  # Sanity check
            size_variants.append(size)
    if size_variants:
        attributes["sizes"] = list(dict.fromkeys(size_variants))

    # Material and fabric - more specific extraction
    material_details = {}
    for key_pattern, selector in [
        (
            "material",
            "[id*='material'], [class*='material'], [id*='fabric'], [class*='fabric']",
        ),
        ("composition", "[id*='composition'], [class*='composition']"),
        (
            "care",
            "[id*='care-instructions'], [class*='care'], [id*='washing'], [class*='washing']",
        ),
    ]:
        elements = soup.select(selector)
        for el in elements:
            text = el.get_text(" ", strip=True)
            if text and len(text) > 3 and len(text) < 500:
                if key_pattern not in material_details:
                    material_details[key_pattern] = []
                material_details[key_pattern].append(text)

    # Look for material info in bullets specifically (common for Amazon)
    material_patterns = [
        r"(\d+)%\s*(cotton|polyester|rayon|viscose|elastane|spandex|wool|nylon)",
        r"Material:?\s*([^\.;]+)",
        r"Fabric:?\s*([^\.;]+)",
        r"Composition:?\s*([^\.;]+)",
    ]

    for bullet in bullets:
        for pattern in material_patterns:
            match = re.search(pattern, bullet, re.I)
            if match:
                if "composition" not in material_details:
                    material_details["composition"] = []
                material_details["composition"].append(match.group(0))
                break

    if material_details:
        attributes["material_details"] = material_details

    # Schema Product
    schema_product = {}
    for obj in jsonld:
        if isinstance(obj, dict):
            typ = obj.get("@type") or obj.get("type")
            if typ and ("product" in str(typ).lower()):
                schema_product.update(obj)

    # Extract offers/price information
    price_info = {}

    # From Schema.org
    if schema_product and "offers" in schema_product:
        offers = schema_product["offers"]
        if isinstance(offers, dict):
            price_info["price"] = offers.get("price")
            price_info["currency"] = offers.get("priceCurrency")
            price_info["availability"] = offers.get("availability")
        elif isinstance(offers, list) and offers:
            price_info["price"] = offers[0].get("price")
            price_info["currency"] = offers[0].get("priceCurrency")
            price_info["availability"] = offers[0].get("availability")

    # Amazon-specific price extraction
    amazon_price_selectors = [
        "#priceblock_ourprice",
        "#priceblock_saleprice",
        ".a-color-price",
        ".a-price .a-offscreen",
        "#price_inside_buybox",
        "#corePrice_feature_div .a-price",
        ".priceToPay span",
        "#price .a-text-price",
    ]

    for selector in amazon_price_selectors:
        price_element = soup.select_one(selector)
        if price_element:
            price_text = price_element.get_text(strip=True)
            if price_text:
                price_info["displayed_price"] = price_text
                break

    # General price elements (as fallback)
    if not price_info.get("displayed_price"):
        price_elements = soup.select(
            "[id*='price'], [class*='price']:not(del), .offer-price, .product-price, .current-price"
        )
        for el in price_elements:
            text = el.get_text(" ", strip=True)
            if text and re.search(r"\d", text) and len(text) < 50:
                # Skip "from" prices or ranges for now
                if "from" not in text.lower() and "-" not in text and "to" not in text:
                    price_info["displayed_price"] = text
                    break

    # Images - enhanced extraction
    images = []

    # Amazon product images - primary
    main_img = soup.select_one(
        "#landingImage, #imgBlkFront, #main-image, #image-block-container img"
    )
    if main_img:
        for attr in ("src", "data-old-hires", "data-a-dynamic-image"):
            val = main_img.get(attr)
            if val:
                if attr == "data-a-dynamic-image" and val.startswith("{"):
                    try:
                        # Amazon stores multiple image sizes in a JSON structure
                        img_urls = json.loads(val).keys()
                        for url in img_urls:
                            images.append(requests.compat.urljoin(base_url, url))
                    except:
                        pass
                else:
                    images.append(requests.compat.urljoin(base_url, val))

    # Amazon product images - gallery
    gallery_images = soup.select("#altImages img, .item.image img, #thumbs-image img")
    for img in gallery_images:
        for attr in ("src", "data-old-hires", "data-a-dynamic-image"):
            val = img.get(attr)
            if val:
                if attr == "data-a-dynamic-image" and val.startswith("{"):
                    try:
                        img_urls = json.loads(val).keys()
                        for url in img_urls:
                            images.append(requests.compat.urljoin(base_url, url))
                    except:
                        pass
                else:
                    images.append(requests.compat.urljoin(base_url, val))

    # Add any image variants
    variant_images = soup.select(
        "#imageBlock .image.item img, #imageBlockVariations img"
    )
    for img in variant_images:
        for attr in ("src", "data-old-hires", "data-a-dynamic-image"):
            val = img.get(attr)
            if val and not val.endswith((".gif", ".svg")):
                images.append(requests.compat.urljoin(base_url, val))

    # Meta tags and schema images
    if metas.get("og:image"):
        images.append(requests.compat.urljoin(base_url, metas.get("og:image")))
    if metas.get("og:image:secure_url"):
        images.append(
            requests.compat.urljoin(base_url, metas.get("og:image:secure_url"))
        )
    schema_img = schema_product.get("image")
    if schema_img:
        if isinstance(schema_img, list):
            for it in schema_img:
                images.append(requests.compat.urljoin(base_url, str(it)))
        else:
            images.append(requests.compat.urljoin(base_url, str(schema_img)))
    link_img = soup.find("link", {"rel": "image_src"})
    if link_img and link_img.get("href"):
        images.append(requests.compat.urljoin(base_url, link_img["href"]))

    # Standard image extraction (as fallback)
    for img in soup.find_all("img"):
        # Skip tiny images and navigation elements
        if img.get("width") and int(img.get("width")) < 50:
            continue

        for attr in (
            "src",
            "data-src",
            "data-old-hires",
            "data-srcset",
            "data-lazy-src",
            "data-zoom-image",
            "data-large",
        ):
            val = img.get(attr)
            if val:
                if "," in val and " " in val:
                    parts = [p.strip().split()[0] for p in val.split(",") if p.strip()]
                    for p in parts:
                        images.append(requests.compat.urljoin(base_url, p))
                else:
                    images.append(requests.compat.urljoin(base_url, val))
        srcset = img.get("srcset") or img.get("data-srcset")
        if srcset:
            parts = [p.strip().split()[0] for p in srcset.split(",") if p.strip()]
            for p in parts:
                images.append(requests.compat.urljoin(base_url, p))

    # Look for hidden large images
    for a in soup.find_all("a", href=True):
        href = a.get("href", "")
        if re.search(r"\.(jpe?g|png|webp|gif)(\?|$)", href, re.I):
            images.append(requests.compat.urljoin(base_url, href))

    # Filter and deduplicate images
    images = [i for i in (images or []) if i and not i.endswith((".ico", ".svg"))]
    # Filter out tracking pixels and tiny images (common in Amazon)
    images = [
        i
        for i in images
        if not re.search(r"(pixel|transparent|spacer|nav-sprite)", i, re.I)
    ]
    # Remove common Amazon UI elements
    images = [
        i
        for i in images
        if not re.search(r"(sprite|logo|gno/sprites|nav-sprite)", i, re.I)
    ]
    images = list(dict.fromkeys(images))[:30]  # Keep more images

    # Build text blob with all extracted information
    details_text = " ".join(f"{k}: {v}" for k, v in (product_details or {}).items())
    jsonld_text = " ".join(json.dumps(o) for o in (jsonld or []) if isinstance(o, dict))
    text_blob = "\n".join(
        filter(
            None,
            [
                title,
                description_meta or schema_product.get("description", ""),
                details_text,
                " ".join(bullets or []),
                " ".join(paras or []),
                " ".join(description_blocks or []),
            ],
        )
    )

    # Ensure consistent return types (avoid None) and include enhanced data
    return {
        "title": title or "",
        "metas": metas or {},
        "jsonld": jsonld or [],
        "schema_product": schema_product or {},
        "js_data": js_data or {},
        "paragraphs": paras or [],
        "bullets": bullets or [],
        "description_blocks": description_blocks or [],
        "product_details": product_details or {},
        "attributes": attributes or {},
        "material_details": material_details or {},
        "price_info": price_info or {},
        "images": images or [],
        "text": text_blob or "",
    }

In [36]:
def test_scrape_extraction(url: str = None):
    import json, os, traceback

    if not url:
        url = "https://www.amazon.com/True-Classic-Mens-T-Shirts-Hipster/dp/B0F4J419WT/ref=sr_1_3_sspa"
    print("Fetching:", url)
    try:
        page = fetch_page(url)
        print("Fetched with:", page.get("fetched_with"), "error:", page.get("error"))
        if not page.get("html"):
            print("No HTML; abort.")
            return
        extracted = extract_text_and_images(page["html"], page["url"])
        # attrs = extract_product_attributes(extracted)
        out = {
            "title": extracted.get("title"),
            "meta_description": extracted.get("metas", {}).get("description"),
            "schema_product_keys": list(extracted.get("schema_product", {}).keys()),
            "first_paragraphs": (extracted.get("paragraphs") or [])[:5],
            "bullets": (extracted.get("bullets") or [])[:8],
            "product_details_preview": {
                k: (extracted.get("product_details") or {}).get(k)
                for k in list((extracted.get("product_details") or {}).keys())[:8]
            },
            "images_count": len(extracted.get("images") or []),
            "sample_images": (extracted.get("images") or [])[:6],
            # "extracted_attributes": attrs,
        }
        print(json.dumps(out, indent=2))
    except Exception:
        print("Error during test_scrape_extraction:")
        traceback.print_exc()

In [ ]:
# url = "https://www.amazon.com/Years-Hustle-Birthday-Gift-Motivational/dp/B0FRXJVTGD/ref=sr_1_7?sr=8-7&psc=1"
url = "https://www.nike.com/t/sportswear-t-shirt-K1saIG0g/HV0180-072"
test_scrape_extraction(url)
# test_detailed_scrape(url, use_playwright=True)

In [ ]:
# pip install playwright beautifulsoup4
# playwright install

import asyncio,nest_asyncio
from playwright.sync_api import sync_playwright
from bs4 import BeautifulSoup
import json
import re
nest_asyncio.apply()
url = "https://www.nike.com/t/sportswear-t-shirt-K1saIG0g/HV0180-072"


# ...existing code...
async def extract_product_data(url):
    """Async playwright fetcher returning (data, html)."""
    from playwright.async_api import async_playwright

    data = {}
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        async def handle_response(response):
            try:
                url_l = response.url.lower()
                if "product" in url_l or "graphql" in url_l:
                    ct = response.headers.get("content-type", "")
                    if "json" in ct:
                        try:
                            json_data = await response.json()
                        except Exception:
                            json_data = None
                        if isinstance(json_data, dict) and (
                            "data" in json_data or "objects" in json_data
                        ):
                            data["api_response"] = json_data
            except Exception:
                pass

        page.on("response", handle_response)
        await page.goto(url, timeout=60000)
        await page.wait_for_load_state("networkidle")
        html = await page.content()
        await browser.close()

    return data, html


def parse_product_info(data, html):
    """Parse the Nike product JSON if present, fallback to HTML"""
    soup = BeautifulSoup(html, "html.parser")
    product = {}

    # Basic HTML scrape
    product["title"] = soup.select_one("h1") and soup.select_one("h1").get_text(
        strip=True
    )
    product["price"] = soup.select_one(
        "div[data-test='product-price']"
    ) and soup.select_one("div[data-test='product-price']").get_text(strip=True)
    product["description"] = soup.select_one(
        "div[data-test='product-description']"
    ) and soup.select_one("div[data-test='product-description']").get_text(
        " ", strip=True
    )
    product["images"] = [
        img["src"] for img in soup.select("img") if "http" in img.get("src", "")
    ]
    product["sizes"] = [
        btn.get_text(strip=True)
        for btn in soup.select("button[data-qa='size-available']")
    ]

    # JSON data
    if "api_response" in data:
        api_json = data["api_response"]
        product["raw_json"] = api_json

        # Optional quick field extraction
        text_dump = json.dumps(api_json).lower()
        for key in ["productid", "stylecolor", "colordescription", "displayname"]:
            match = re.search(rf'"{key}"\s*:\s*"([^"]+)"', text_dump)
            if match:
                product[key] = match.group(1)

    return product

async def main():
    data, html = await extract_product_data(url)
    # product = parse_product_info(data, html)
    # print(json.dumps(product, indent=2))


In [16]:
def extract_materials(text_blob: str, context_chars: int = 80) -> List[Dict[str, Any]]:
    """
    Find material/composition mentions and return structured matches:
    - raw: matched string
    - pattern: which pattern matched
    - context: surrounding text
    - components: list of {material, percent (int|None)}
    """
    import re, json

    materials = []
    text = text_blob or ""

    # Patterns to capture common forms
    patterns = {
        "percent_material": re.compile(
            r"(\d{1,3})%\s*(cotton|polyester|rayon|viscose|elastane|spandex|wool|nylon|linen|silk)",
            re.I,
        ),
        "percent_list": re.compile(
            r"((?:\d{1,3}%\s*(?:cotton|polyester|rayon|viscose|elastane|spandex|wool|nylon|linen|silk)\s*,?\s*)+)",
            re.I,
        ),
        "composition_field": re.compile(r"composition[:\s]*([^;\n\r]+)", re.I),
        "material_field": re.compile(r"material[:\s]*([^;\n\r]+)", re.I),
        "made_of": re.compile(r"(made (?:of|from|with)\s+[^.;\n\r]{1,120})", re.I),
    }

    def context_for(m):
        s = max(0, m.start() - context_chars)
        e = min(len(text), m.end() + context_chars)
        return text[s:e].replace("\n", " ")

    # helper to parse a composition fragment into components
    comp_split_re = re.compile(r"[,&/]| and ")
    material_word_re = re.compile(
        r"(?:\d{1,3}%\s*)?(cotton|polyester|rayon|viscose|elastane|spandex|wool|nylon|linen|silk)",
        re.I,
    )

    def parse_components(fragment: str):
        parts = [p.strip() for p in re.split(r"[;,/]| and | & ", fragment) if p.strip()]
        comps = []
        for p in parts:
            # try to capture percent + material
            m = material_word_re.search(p)
            if m:
                mat = m.group(1).lower()
                pct_m = re.search(r"(\d{1,3})\s*%", p)
                pct = int(pct_m.group(1)) if pct_m else None
                comps.append({"material": mat, "percent": pct, "raw": p})
            else:
                # fallback: include whole token as raw
                comps.append({"material": None, "percent": None, "raw": p})
        return comps

    # 1) explicit percent-material lists
    for m in patterns["percent_list"].finditer(text):
        raw = m.group(1).strip()
        mats = []
        for mm in patterns["percent_material"].finditer(raw):
            mats.append(
                {
                    "material": mm.group(2).lower(),
                    "percent": int(mm.group(1)),
                    "raw": mm.group(0),
                }
            )
        materials.append(
            {
                "raw": raw,
                "pattern": "percent_list",
                "context": context_for(m),
                "components": mats or parse_components(raw),
            }
        )

    # 2) composition: field
    for m in patterns["composition_field"].finditer(text):
        raw = m.group(1).strip()
        materials.append(
            {
                "raw": raw,
                "pattern": "composition_field",
                "context": context_for(m),
                "components": parse_components(raw),
            }
        )

    # 3) material: field
    for m in patterns["material_field"].finditer(text):
        raw = m.group(1).strip()
        materials.append(
            {
                "raw": raw,
                "pattern": "material_field",
                "context": context_for(m),
                "components": parse_components(raw),
            }
        )

    # 4) made of / made from
    for m in patterns["made_of"].finditer(text):
        raw = m.group(0).strip()
        # try to extract comma-separated materials within the phrase
        phrase = re.sub(r"^made (?:of|from|with)\s+", "", raw, flags=re.I)
        materials.append(
            {
                "raw": raw,
                "pattern": "made_of",
                "context": context_for(m),
                "components": parse_components(phrase),
            }
        )

    # 5) fallback single percent-material mentions not part of list
    for m in patterns["percent_material"].finditer(text):
        raw = m.group(0).strip()
        materials.append(
            {
                "raw": raw,
                "pattern": "percent_material",
                "context": context_for(m),
                "components": [
                    {
                        "material": m.group(2).lower(),
                        "percent": int(m.group(1)),
                        "raw": raw,
                    }
                ],
            }
        )

    # dedupe by raw text (preserve order)
    seen = set()
    out = []
    for it in materials:
        key = it["raw"]
        if key not in seen:
            seen.add(key)
            out.append(it)
    return out

In [4]:
def download_best_image(image_urls: List[str], timeout=6) -> Optional[bytes]:
    best = None
    best_area = 0
    for url in image_urls:
        try:
            r = requests.get(
                url,
                timeout=timeout,
                stream=True,
                headers={"User-Agent": "vr-mall-bot/1.0"},
            )
            r.raise_for_status()
            content = r.content
            im = Image.open(io.BytesIO(content)).convert("RGB")
            w, h = im.size
            area = w * h
            if area > best_area:
                best_area = area
                best = content
        except Exception:
            continue
    return best


def make_thumbnail_and_color(img_bytes: bytes, thumb_size=(256, 256)):
    im = Image.open(io.BytesIO(img_bytes)).convert("RGB")
    thumb = im.copy()
    thumb.thumbnail(thumb_size, Image.LANCZOS)
    buf = io.BytesIO()
    thumb.save(buf, format="PNG")
    b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
    # compute simple dominant (average) color
    small = im.resize((32, 32))
    arr = np.array(small).reshape(-1, 3).astype(np.float32)
    avg = arr.mean(axis=0) / 255.0
    return {
        "thumbnail_base64": b64,
        "dominant_color": [float(avg[0]), float(avg[1]), float(avg[2])],
    }

In [4]:
def caption_image(img_bytes: bytes) -> Optional[str]:
    try:
        from transformers import BlipProcessor, BlipForConditionalGeneration
        import torch

        processor = BlipProcessor.from_pretrained(
            "Salesforce/blip-image-captioning-base"
        )
        model = BlipForConditionalGeneration.from_pretrained(
            "Salesforce/blip-image-captioning-base"
        )
        from PIL import Image as PILImage

        img = PILImage.open(io.BytesIO(img_bytes)).convert("RGB")
        inputs = processor(images=img, return_tensors="pt")
        out = model.generate(**inputs)
        caption = processor.decode(out[0], skip_special_tokens=True)
        return caption
    except Exception:
        return None

In [5]:
def load_embedding_model(name="all-MiniLM-L6-v2"):
    from sentence_transformers import SentenceTransformer

    return SentenceTransformer(name)


def build_faiss_index(embeddings: List[np.ndarray]):
    try:
        import faiss

        dim = embeddings[0].shape[0]
        idx = faiss.IndexFlatL2(dim)
        idx.add(np.vstack(embeddings))
        return idx
    except Exception as e:
        print("faiss not available:", e)
        return None


def embed_texts(model, texts: List[str]) -> List[np.ndarray]:
    return model.encode(texts, show_progress_bar=False, convert_to_numpy=True)

In [6]:
def serpapi_search(query: str, num=5):
    if not SERPAPI_KEY or SERPAPI_KEY.startswith("<"):
        print("SERPAPI_KEY not set - skipping web search")
        return []
    params = {"engine": "google", "q": query, "api_key": SERPAPI_KEY, "num": num}
    r = requests.get("https://serpapi.com/search.json", params=params, timeout=10)
    r.raise_for_status()
    data = r.json()
    results = []
    for item in data.get("organic_results", [])[:num]:
        link = item.get("link")
        title = item.get("title")
        snippet = item.get("snippet")
        results.append({"url": link, "title": title, "snippet": snippet})
    return results

In [22]:
def extract_product_attributes(source: Any) -> Dict[str, Any]:
    """
    Extract structured product attributes. 'source' can be a text string or the dict returned by
    extract_text_and_images (recommended).
    """
    import re, json

    if isinstance(source, dict):
        text = source.get("text", "") or ""
        details = source.get("product_details", {}) or {}
        schema = source.get("schema_product", {}) or {}
        bullets = " ".join(source.get("bullets", []) or [])
        combined = " ".join(
            [text, bullets, " ".join(details.values()), json.dumps(schema or {})]
        )
    else:
        combined = str(source or "")

    txt = combined.lower()

    attributes = {
        "material": None,
        "material_components": [],  # structured components from extract_materials
        "fabric_type": None,
        "weight": None,
        "care": None,
        "brand": None,
        "sku": None,
        "color": None,
        "other": {},
    }

    # Patterns (existing)
    material_patterns = [
        r"(\d+)%\s*(cotton|polyester|rayon|viscose|elastane|spandex|wool|nylon|linen|silk)",
        r"(cotton|polyester|rayon|viscose|elastane|spandex|wool|nylon|linen|silk)\s*blend",
        r"made (?:of|from|with)\s+(cotton|polyester|rayon|viscose|wool|nylon|linen|silk)",
        r"material[:\s]*?(cotton|polyester|rayon|viscose|wool|nylon|linen|silk)",
        r"composition[:\s]*?([^\n,;]+)",
    ]
    fabric_patterns = [
        r"(jersey|knit|woven|terry|fleece|pique|oxford|poplin|twill|denim|canvas|corduroy|flannel|satin|chiffon|leather)",
        r"fabric[:\s]*?(jersey|knit|woven|terry|fleece|pique|oxford|poplin|twill|denim|canvas)",
    ]
    care_patterns = [
        r"(machine wash|hand wash|dry clean|tumble dry|air dry|iron|do not bleach|do not tumble dry)",
        r"care[:\s]*?(machine wash|hand wash|dry clean|do not bleach|do not tumble dry)",
    ]
    weight_patterns = [
        r"(\d+)\s*(?:gsm|g\/m2|g\/m²|grams|oz)",
        r"weight[:\s]*?(\d+)\s*(?:gsm|g\/m2|grams|oz)",
    ]
    brand_patterns = [r"brand[:\s]*([a-z0-9\-\_ ]{2,50})", r"by\s+([A-Z][\w\- ]{1,50})"]
    sku_patterns = [r"sku[:\s]*([A-Z0-9\-]+)", r"model number[:\s]*([A-Z0-9\-]+)"]

    def find_first(patterns):
        for p in patterns:
            m = re.search(p, txt)
            if m:
                # return group 1 or full match
                return next((g for g in m.groups() if g), m.group(0))
        return None

    # Try schema fields first
    try:
        # schema may contain material/composition, brand, sku
        if isinstance(
            schema := (
                json.loads(source.get("jsonld")[0])
                if isinstance(source, dict)
                and source.get("jsonld")
                and isinstance(source.get("jsonld")[0], str)
                else {}
            ),
            dict,
        ):
            pass
    except Exception:
        pass

    # From schema dict if present in source
    if isinstance(source, dict):
        schema = source.get("schema_product", {}) or {}
        # brand
        b = None
        if isinstance(schema.get("brand"), dict):
            b = schema.get("brand").get("name")
        elif schema.get("brand"):
            b = schema.get("brand")
        if b:
            attributes["brand"] = b
        # sku
        if schema.get("sku"):
            attributes["sku"] = schema.get("sku")
        # material-like fields
        for key in ("material", "materialComposition", "hasMaterial", "isMadeOf"):
            val = schema.get(key)
            if val:
                attributes["material"] = val if not isinstance(val, list) else val
        # description
        if not txt and schema.get("description"):
            txt = (schema.get("description") or "").lower()

    # Use the extractor that returns structured components (if available)
    try:
        mats = extract_materials(combined, context_chars=80)
        if mats:
            # attach structured components and normalize a concise material list
            attributes["material_components"] = mats
            # build a normalized list of material names (preserve percents if present)
            normalized = []
            for entry in mats:
                for comp in entry.get("components", []):
                    mat = comp.get("material")
                    pct = comp.get("percent")
                    if mat:
                        if pct is not None:
                            normalized.append(f"{pct}% {mat}")
                        else:
                            normalized.append(mat)
                    else:
                        # fallback to raw token
                        raw = comp.get("raw")
                        if raw:
                            normalized.append(raw)
            if normalized:
                # dedupe preserving order
                seen = set()
                out = []
                for it in normalized:
                    if it not in seen:
                        seen.add(it)
                        out.append(it)
                attributes["material"] = out
    except Exception:
        # safe fallback to regex-based heuristics below
        pass

    # fallback regex from combined text (kept for backward compatibility)
    mat = []
    for p in material_patterns:
        for m in re.findall(p, txt):
            if isinstance(m, tuple):
                for part in m:
                    if part and not part.isdigit():
                        mat.append(part.strip())
            else:
                if m and not str(m).isdigit():
                    mat.append(m.strip())
    if mat and not attributes.get("material"):
        attributes["material"] = list(dict.fromkeys(mat))

    # fabric type
    f = re.findall("|".join(fabric_patterns), txt)
    if f:
        try:
            # flatten tuples if present
            flat = []
            for item in f:
                if isinstance(item, tuple):
                    flat.extend([it for it in item if it])
                else:
                    flat.append(item)
            attributes["fabric_type"] = list(dict.fromkeys(flat))
        except Exception:
            attributes["fabric_type"] = list(dict.fromkeys(f))

    # care
    c = re.findall("|".join(care_patterns), txt)
    if c:
        attributes["care"] = list(
            dict.fromkeys([cc if not isinstance(cc, tuple) else cc[0] for cc in c])
        )

    # weight
    w = find_first(weight_patterns)
    if w:
        attributes["weight"] = w

    # brand and sku via regex fallback
    if not attributes.get("brand"):
        br = find_first(brand_patterns)
        if br:
            attributes["brand"] = br.strip()
    if not attributes.get("sku"):
        sk = find_first(sku_patterns)
        if sk:
            attributes["sku"] = sk.strip()

    # Colors
    col = re.findall(r"(?:color|colour)[:\s]*([a-z0-9 \-]+)", txt)
    if col:
        attributes["color"] = list(dict.fromkeys([c.strip() for c in col]))

    # Add any matched product_details keys that look useful
    if isinstance(source, dict):
        for k, v in (source.get("product_details") or {}).items():
            kl = k.lower()
            if "material" in kl or "fabric" in kl or "composition" in kl:
                attributes["material"] = attributes.get("material") or v
            if "care" in kl or "wash" in kl:
                attributes["care"] = attributes.get("care") or v
            if "weight" in kl or "gsm" in kl:
                attributes["weight"] = attributes.get("weight") or v
            if "brand" in kl and not attributes.get("brand"):
                attributes["brand"] = v
            if ("model" in kl or "sku" in kl or "asin" in kl) and not attributes.get(
                "sku"
            ):
                attributes["sku"] = attributes.get("sku") or v

    # final normalization: lists -> strings occasionally, keep lists for material/care/fabric
    return attributes

In [17]:
def process_product_url(url: str, do_search=True, search_top_n=4):
    # 1) fetch primary page
    page = fetch_page(url)
    if page.get("error"):
        return {"ok": False, "error": page.get("error", "Failed to fetch page")}
    extracted = extract_text_and_images(page["html"], page["url"])

    # Extract product attributes from the main page text
    product_attributes = extract_product_attributes(extracted["text"])

    # Initialize collections for search results and documents
    serp = []
    docs = []
    embeddings = []
    faiss_idx = None

    # Add the primary page to documents
    docs.append({"source": url, "text": extracted["text"], "title": extracted["title"]})

    # If material info not found or incomplete, enhance with additional searches
    material_results = []
    if not product_attributes.get("material") and do_search:
        material_query = modify_serpapi_for_materials(
            extracted["title"] + " material composition", extracted["title"]
        )
        material_results = serpapi_search(material_query, num=2)
        # Process material search results
        for r in material_results:
            try:
                p = fetch_page(r["url"])
                if p.get("error"):
                    continue
                material_text = extract_text_and_images(p.get("html", ""), r["url"])[
                    "text"
                ]
                material_attrs = extract_product_attributes(material_text)
                # Add to documents for embeddings
                docs.append(
                    {
                        "source": r["url"],
                        "text": material_text,
                        "title": r.get("title", ""),
                    }
                )
                # Merge any new material attributes found
                for key, value in material_attrs.items():
                    if value and not product_attributes.get(key):
                        product_attributes[key] = value
            except Exception as e:
                print(f"Error processing material result: {e}")
                continue

    # Run general search if requested
    if do_search:
        try:
            search_query = extracted["title"]
            serp = serpapi_search(search_query, num=search_top_n)

            # Process general search results
            for r in serp:
                if r["url"] != url:  # Skip the original URL
                    try:
                        p = fetch_page(r["url"])
                        if p.get("error"):
                            continue
                        result_text = extract_text_and_images(
                            p.get("html", ""), r["url"]
                        )
                        docs.append(
                            {
                                "source": r["url"],
                                "text": result_text.get("text", ""),
                                "title": r.get("title", ""),
                            }
                        )
                    except Exception as e:
                        print(f"Error processing search result: {e}")
        except Exception as e:
            print(f"Error during general search: {e}")

    # Process images
    best_img_bytes = download_best_image(extracted["images"])
    thumb_info = {}
    if best_img_bytes:
        thumb_info = make_thumbnail_and_color(best_img_bytes)
        caption = caption_image(best_img_bytes) or None
    else:
        caption = None

    # Generate embeddings if we have documents
    if docs:
        try:
            emb_model = load_embedding_model()
            texts = [d["title"] + "\n" + d["text"] for d in docs]
            embeddings = embed_texts(emb_model, texts)
            faiss_idx = build_faiss_index(embeddings)
        except Exception as e:
            print(f"Error generating embeddings: {e}")

    # Add the extracted product attributes to the result
    result = {
        "ok": True,
        "url": url,
        "title": extracted["title"],
        "text": extracted["text"],
        "images": extracted["images"],
        "thumbnail_base64": thumb_info.get("thumbnail_base64"),
        "dominant_color": thumb_info.get("dominant_color"),
        "caption": caption,
        "product_attributes": product_attributes,  # Add the material info here
        "related_pages": [
            {"url": r["url"], "title": r.get("title"), "snippet": r.get("snippet")}
            for r in (serp if do_search else [])
        ],
        "docs": docs,
        "embeddings_count": len(embeddings) if embeddings is not None else 0,
        "faiss_index": "created" if faiss_idx is not None else None,
    }
    return result

In [18]:
url = "https://www.amazon.com/True-Classic-Mens-T-Shirts-Hipster/dp/B0F4J419WT/ref=sr_1_3_sspa?sr=8-3-spons&sp_csd=d2lkZ2V0TmFtZT1zcF9hdGY&psc=1"
res = process_product_url(url, do_search=True, search_top_n=3)
print(
    json.dumps(
        {
            k: (
                v
                if k not in ("thumbnail_base64", "docs")
                else (v if k == "docs" else "[thumbnail]")
            )
            for k, v in res.items()
        },
        indent=2,
    )
)

{
  "ok": true,
  "url": "https://www.amazon.com/True-Classic-Mens-T-Shirts-Hipster/dp/B0F4J419WT/ref=sr_1_3_sspa?sr=8-3-spons&sp_csd=d2lkZ2V0TmFtZT1zcF9hdGY&psc=1",
  "title": "True Classic Mens T-Shirts \u2013 Curved Hem Tee Mens, Hipster Shirts for Men, Long Tail T Shirts for Man, Drop Cut Lounge/Sleep, Pack of 4, Variety, XL at Amazon Men\u2019s Clothing store",
  "text": "True Classic Mens T-Shirts \u2013 Curved Hem Tee Mens, Hipster Shirts for Men, Long Tail T Shirts for Man, Drop Cut Lounge/Sleep, Pack of 4, Variety, XL at Amazon Men\u2019s Clothing store\nWe offer easy, convenient returns with at least one free return option: no shipping charges. All returns must comply with our returns policy.\nThis item has been tested to certify it can ship safely in its original box or bag to avoid unnecessary packaging. Since 2015, we have reduced the weight of outbound packaging per shipment by 41% on average, that\u2019s over 2 million tons of packaging material.\nCrew Neck Tees\nCurved 